# Output Head Architecture & Ordinal Regression Ablation Benchmark (Hypothesis H3)

[![PyTorch](https://img.shields.io/badge/PyTorch-2.0%2B-EE4C2C.svg?logo=pytorch)](https://pytorch.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![Domain: Medical Imaging](https://img.shields.io/badge/Domain-Medical%20Computer%20Vision-success.svg)]()
[![Task: 5-Grade Ordinal DR](https://img.shields.io/badge/Task-5--Grade%20Ordinal%20DR-informational.svg)]()

---

## Executive Summary & Scientific Methodology

This notebook implements the systematic **5-Round End-to-End Head Ablation Benchmark** evaluating 5 output loss formulations on a standardized **ResNet-50** backbone for 5-class Diabetic Retinopathy (DR) grading:
1. **Nominal Softmax + Categorical Cross-Entropy (Softmax_CE)**: Standard classification baseline.
2. **Nominal Softmax + Continuous QWK Loss (Softmax_QWK)**: Softmax with quadratic penalty optimization.
3. **Consistent Rank Logits (CORAL)**: Shared linear weights with independent threshold biases $b_0..b_3$.
4. **Conditional Ordinal Regression Neural Network (CORN)**: Conditional probability formulation $P(y > k \mid y \ge k)$.
5. **Cumulative Link Model + Continuous QWK Loss (CLM_QWK - Champion)**: Complementary log-log ($\text{clog-log}$) link function with strictly monotonic cutpoints $\theta_1 < \theta_2 < \theta_3 < \theta_4$ and dynamic QWK loss warmup.

### Controlled Experimental Protocol:
- **Strictly Controlled Backbone**: All 5 heads are paired with identical **ResNet-50** architectures ($300 \times 300$ resolution, identical Green-channel Ben Graham preprocessing, identical AdamW optimizer, and Cosine Annealing schedule).
- **Independent End-to-End Optimization**: Each head is trained from scratch together with its own ResNet-50 backbone across 5 distinct training loops (zero shared latent caching, zero inter-head bias).
- **Zero Patient Leakage**: Evaluation is performed on a leak-free 15% In-Domain Patient-Grouped Held-out Test Set.
- **Core Hypothesis H3 Verification**: Verifies rank monotonicity violation rate (target: $\mathbf{0.0\%}$) and test QWK improvement (target: $\mathbf{+0.041}$ gain over Softmax_CE).

---

## Table of Contents

- [1. Environment Setup & Publication Configuration](#1-environment-setup--publication-configuration)
  - [1.1 Library Imports & Singleton Logger](#11-library-imports--singleton-logger)
  - [1.2 Publication-Ready Plotting Standards](#12-publication-ready-plotting-standards)
  - [1.3 Dataset Path Definitions & Execution Mode](#13-dataset-path-definitions--execution-mode)
- [2. Multi-Center Dataset Ingestion & Patient-Level Splitting](#2-multi-center-dataset-ingestion--patient-level-splitting)
  - [2.1 Patient ID Parsing & Zero-Leakage Grouping](#21-patient-id-parsing--zero-leakage-grouping)
  - [2.2 Master Cohort Ingestion & DataLoader Initialization](#22-master-cohort-ingestion--dataloader-initialization)
- [3. Deep Neural Network Architecture & 5 Output Heads](#3-deep-neural-network-architecture--5-output-heads)
  - [3.1 Shared ResNet-50 Feature Extractor](#31-shared-resnet-50-feature-extractor)
  - [3.2 Output Head 1: Softmax + Categorical Cross-Entropy (Softmax_CE)](#32-output-head-1-softmax--categorical-cross-entropy-softmax_ce)
  - [3.3 Output Head 2: Softmax + Continuous QWK Loss (Softmax_QWK)](#33-output-head-2-softmax--continuous-qwk-loss-softmax_qwk)
  - [3.4 Output Head 3: Consistent Rank Logits (CORAL)](#34-output-head-3-consistent-rank-logits-coral)
  - [3.5 Output Head 4: Conditional Ordinal Regression Neural Network (CORN)](#35-output-head-4-conditional-ordinal-regression-neural-network-corn)
  - [3.6 Output Head 5: Cumulative Link Model + Continuous QWK (CLM_QWK - Champion)](#36-output-head-5-cumulative-link-model--continuous-qwk-clm_qwk---champion)
  - [3.7 Unified ResNet50AblationModel & Factory Builder](#37-unified-resnet50ablationmodel--factory-builder)
- [4. Clinical Metrics & PyTorch Training Engine](#4-clinical-metrics--pytorch-training-engine)
  - [4.1 Ophthalmology Screening & Statistical Metrics](#41-ophthalmology-screening--statistical-metrics)
  - [4.2 High-Performance DRTrainer Engine with Mixed Precision](#42-high-performance-drtrainer-engine-with-mixed-precision)
- [5. Systematic 5-Round End-to-End Ablation Benchmark](#5-systematic-5-round-end-to-end-ablation-benchmark)
  - [5.1 Multi-Round Training Execution & Checkpoint Tracking](#51-multi-round-training-execution--checkpoint-tracking)
  - [5.2 Results Compilation & CSV/JSON Export](#52-results-compilation--csvjson-export)
- [6. Publication-Quality Head Ablation Visualization Suite](#6-publication-quality-head-ablation-visualization-suite)
  - [6.1 Figure 1: Multi-Metric Clinical Comparison across 5 Heads](#61-figure-1-multi-metric-clinical-comparison-across-5-heads)
  - [6.2 Figure 2: Monotonicity Violations vs QWK Tradeoff](#62-figure-2-monotonicity-violations-vs-qwk-tradeoff)
  - [6.3 Figure 3: Per-Grade Sensitivity Breakdown across 5 Heads](#63-figure-3-per-grade-sensitivity-breakdown-across-5-heads)
  - [6.4 Figure 4: 1D Latent Severity Spectrum & Monotonic Cutpoints of CLM_QWK](#64-figure-4-1d-latent-severity-spectrum--monotonic-cutpoints-of-clm_qwk)
  - [6.5 Figure 5: Referable DR ROC Curves across 5 Heads](#65-figure-5-referable-dr-roc-curves-across-5-heads)

## 1. Environment Setup & Configuration

Initializes deep learning libraries, singleton logging infrastructure, academic plotting styles, and path configurations.

In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path
from typing import List, Tuple, Union, Dict, Optional, Any

# --- Deep Learning & PyTorch Core ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

# --- Scientific Computing & Image Processing ---
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix,
    accuracy_score,
    recall_score,
    roc_curve,
    auc,
)

# --- Visualization & Academic Plotting ---
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Patch, Rectangle, FancyBboxPatch

# Setup module logger with singleton StreamHandler
logger = logging.getLogger("HeadAblation")
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("[%(levelname)s | %(asctime)s | HeadAblation] %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

# Set random seed for reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# ==============================================================================
# 0. GLOBAL PUBLICATION STYLE CONFIGURATION (IEEE / Springer / Nature Standards)
# ==============================================================================
def configure_publication_style():
    """Applies cohesive typography, grid rules, and high-contrast academic aesthetics."""
    plt.rcParams.update({
        'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Liberation Sans'],
        'font.family': 'sans-serif',
        'font.size': 10.5,
        'axes.labelsize': 11.5,
        'axes.titlesize': 12.5,
        'axes.titleweight': 'bold',
        'axes.labelweight': 'bold',
        'xtick.labelsize': 10.0,
        'ytick.labelsize': 10.0,
        'legend.fontsize': 9.5,
        'legend.title_fontsize': 10.5,
        'figure.titlesize': 14.0,
        'figure.titleweight': 'bold',
        'axes.linewidth': 1.1,
        'axes.edgecolor': '#2d3748',
        'grid.linewidth': 0.7,
        'grid.alpha': 0.45,
        'grid.linestyle': '--',
        'grid.color': '#cbd5e1',
        'mathtext.fontset': 'dejavusans',
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.08
    })
    sns.set_theme(style="whitegrid", rc={'grid.linestyle': '--', 'grid.alpha': 0.45})

configure_publication_style()

# Academic Color Palette Definitions
C_NAVY     = "#1a365d"  # Baseline / Softmax_CE Primary
C_TEAL     = "#0d9488"  # Proposed Champion Model (CLM_QWK)
C_ORANGE   = "#e65100"  # CORAL / Ordinal Highlight
C_CRIMSON  = "#c53030"  # Severe Monotonicity Violations / Drops
C_PURPLE   = "#6b21a8"  # CLM Monotonic Decision Cutpoints
C_GRAY     = "#94a3b8"  # Secondary / Neutral Bars
C_DARKGRAY = "#475569"  # Annotations
C_EMERALD  = "#16a34a"  # High Accuracy / Compliance
C_AMBER    = "#d97706"  # CORN / Softmax_QWK Highlights

In [ ]:
# ==============================================================================
# 1.2 CONFIGURATION & PATH RESOLUTION (SMOKE_TEST vs. FULL_RUN)
# ==============================================================================
# Set SMOKE_TEST = False for official full training run on all 5 heads
SMOKE_TEST = True

IMG_SIZE = 300
BATCH_SIZE = 8
QWK_WARMUP_EPOCHS = 3
ABLATION_EPOCHS = 2 if SMOKE_TEST else 12

CHECKPOINT_DIR_HEAD = "./checkpoints_head_ablation"
RESULTS_DIR_HEAD = "./results/head_ablation"
FIGURES_DIR = "./results/figures"
os.makedirs(CHECKPOINT_DIR_HEAD, exist_ok=True)
os.makedirs(RESULTS_DIR_HEAD, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# 5 Output Heads for Systematic Ablation
EVALUATED_HEADS = ["Softmax_CE", "Softmax_QWK", "CORAL", "CORN", "CLM_QWK"]

print(f"Evaluated Output Heads: {EVALUATED_HEADS}")
print(f"Execution Mode: {'SMOKE TEST (Rapid Sanity Check: 2 Epochs)' if SMOKE_TEST else 'FULL END-TO-END ABLATION (12 Epochs)'}")

## 2. Multi-Center Dataset Ingestion & Patient-Level Splitting

This section loads the multi-center dataset, enforces zero-leakage patient-level grouping, applies Green-channel Ben Graham preprocessing, and builds the PyTorch DataLoaders.

In [ ]:
# ==============================================================================
# 2.1 PATIENT ID PARSING & MASTER DATASET INGESTION
# ==============================================================================
def parse_patient_and_side(image_id: str, dataset_key: str) -> Tuple[str, str]:
    """Extracts patient_id and eye laterality to prevent cross-split patient leakage."""
    img_str = str(image_id).lower()
    k = dataset_key.lower()
    side = "unknown"
    if "left" in img_str or "_l" in img_str:
        side = "left"
    elif "right" in img_str or "_r" in img_str:
        side = "right"

    if "idrid" in k:
        base = img_str.replace(".jpg", "").replace(".png", "")
        pid = base.split("_")[-1]
    elif "eyepacs" in k or "deepdrid" in k:
        pid = img_str.split("_")[0]
    elif "ddr" in k:
        parts = img_str.replace(".jpg", "").replace(".png", "").split("-")
        pid = "-".join(parts[:2]) if len(parts) >= 2 else img_str.split(".")[0]
    elif "messidor" in k:
        pid = img_str.split("_")[0]
    else:
        pid = img_str
    return f"{dataset_key}::{pid}", side


def load_dataset_metadata(
    data_dir: str = "../../../../data/raw/hybrid",
    smoke_test: bool = False,
    samples_per_class: Optional[Dict[str, int]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
  """Loads train, validation, and test metadata splits.

  Args:
      data_dir: Directory path containing the split metadata CSV files.
      smoke_test: Flag to enable fast pipeline smoke testing with reduced
        samples.
      samples_per_class: Number of samples per class/diagnosis when
        smoke_test=True.

  Returns:
      Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: (train_df, val_df,
      test_df).
  """
  if samples_per_class is None:
    samples_per_class = {"train": 15, "val": 8, "test": 10}

  splits = ["train", "val", "test"]
  paths = {s: os.path.join(data_dir, f"{s}_metadata.csv") for s in splits}

  # Fail-fast check to ensure all required metadata files exist
  for split_name, file_path in paths.items():
    if not os.path.exists(file_path):
      raise FileNotFoundError(
          f"[ERROR] Missing metadata file for split '{split_name}': {file_path}"
      )

  # Load raw split metadata
  train_df = pd.read_csv(paths["train"])
  val_df = pd.read_csv(paths["val"])
  test_df = pd.read_csv(paths["test"])

  # Downsample per class if smoke test mode is enabled
  if smoke_test:
    print(
        "[INFO] SMOKE TEST = TRUE -> Downsampling datasets across all classes."
    )
    train_df = (
        train_df.groupby("diagnosis")
        .head(samples_per_class["train"])
        .reset_index(drop=True)
    )
    val_df = (
        val_df.groupby("diagnosis")
        .head(samples_per_class["val"])
        .reset_index(drop=True)
    )
    test_df = (
        test_df.groupby("diagnosis")
        .head(samples_per_class["test"])
        .reset_index(drop=True)
    )

  mode_label = "SMOKE TEST" if smoke_test else "FULL DATASET"
  print(
      f"[{mode_label}] Train: {len(train_df):,} | Val: {len(val_df):,} | Test:"
      f" {len(test_df):,}"
  )

  return train_df, val_df, test_df

train_metadata, val_metadata, test_metadata = load_dataset_metadata(
    data_dir="../../../../data/raw/hybrid",
    smoke_test=SMOKE_TEST
)

In [ ]:
# ==============================================================================
# 2.2 PREPROCESSING, AUGMENTATIONS, & PYTORCH DATASET
# ==============================================================================
def apply_green_channel_ablation(img_rgb: np.ndarray, ablation_mode: str = "ben_graham_green") -> np.ndarray:
    """Enhances green-channel optical contrast for vascular and lesion visibility."""
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    denoised = cv2.bilateralFilter(bgr, d=9, sigmaColor=75, sigmaSpace=75)
    b, g, r = cv2.split(denoised)

    if ablation_mode == "ben_graham_green":
        blurred = cv2.GaussianBlur(g, (0, 0), 30)
        adjusted = cv2.addWeighted(g, 4, blurred, -4, 128)
        processed = cv2.merge([adjusted, adjusted, adjusted])
    elif ablation_mode == "clahe_green":
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_g = clahe.apply(g)
        processed = cv2.merge([enhanced_g, enhanced_g, enhanced_g])
    else:
        processed = cv2.merge([g, g, g])

    return cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)


def get_transforms(img_size: int = IMG_SIZE, is_train: bool = True) -> transforms.Compose:
    """Standard PyTorch torchvision data augmentations for ResNet-50 feature extractor."""
    if is_train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


class ResNetFundusDataset(Dataset):
    """PyTorch Dataset serving standardized 300x300 fundus images for ResNet-50."""
    def __init__(self, df: pd.DataFrame, img_size: int = IMG_SIZE, split: str = "train", ablation_mode: str = "ben_graham_green"):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.split = split
        self.ablation_mode = ablation_mode
        self.transform = get_transforms(img_size, is_train=(split == "train"))
        self.records = self.df.to_dict("records")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.records[idx]
        label = int(row["diagnosis"])
        img_path = row.get("native_path", "") or row.get("raw_full_path", "")

        if os.path.exists(img_path):
            img_bgr = cv2.imread(img_path)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2BGR) if img_bgr is not None else np.full((512, 512, 3), 128, dtype=np.uint8)
        else:
            img_rgb = np.full((512, 512, 3), fill_value=128, dtype=np.uint8)

        processed_rgb = apply_green_channel_ablation(img_rgb, self.ablation_mode)
        tensor_img = self.transform(processed_rgb)

        return {
            "image": tensor_img,
            "label": label,
            "patient_id": row.get("patient_id", "unknown"),
            "dataset_key": row.get("dataset_key", "unknown")
        }

# Instantiate DataLoaders
train_dataset = ResNetFundusDataset(train_metadata, img_size=IMG_SIZE, split="train")
val_dataset   = ResNetFundusDataset(val_metadata, img_size=IMG_SIZE, split="val")
test_dataset  = ResNetFundusDataset(test_metadata, img_size=IMG_SIZE, split="test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f"DataLoaders Initialized: Train batches={len(train_loader)}, Val batches={len(val_loader)}, Test batches={len(test_loader)}")

## 3. Deep Neural Network Architecture & 5 Output Heads

Defines the shared pre-trained ResNet-50 backbone and the 5 output head modules (`Softmax_CE`, `Softmax_QWK`, `CORAL`, `CORN`, `CLM_QWK`).

In [ ]:
# ==============================================================================
# 3.1 SHARED RESNET-50 BACKBONE & 5 OUTPUT HEAD MODULES
# ==============================================================================
class ResNet50Backbone(nn.Module):
    """Standard pre-trained ResNet-50 feature extractor projecting 2048-d -> 512-d latent."""
    def __init__(self, pretrained: bool = True, latent_dim: int = 512):
        super(ResNet50Backbone, self).__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        base_resnet = models.resnet50(weights=weights)
        self.conv1 = base_resnet.conv1
        self.bn1 = base_resnet.bn1
        self.relu = base_resnet.relu
        self.maxpool = base_resnet.maxpool
        self.layer1 = base_resnet.layer1
        self.layer2 = base_resnet.layer2
        self.layer3 = base_resnet.layer3
        self.layer4 = base_resnet.layer4
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.projection = nn.Sequential(
            nn.Linear(2048, latent_dim),
            nn.Mish(inplace=True),
            nn.LayerNorm(latent_dim)
        )
        self.latent_dim = latent_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        feat = torch.flatten(self.gap(x), 1)
        return self.projection(feat)


class ContinuousQWKLoss(nn.Module):
    """Differentiable Continuous Quadratic Weighted Kappa Loss."""
    def __init__(self, num_classes: int = 5, eps: float = 1e-7):
        super(ContinuousQWKLoss, self).__init__()
        self.num_classes = num_classes
        self.eps = eps
        w = torch.zeros((num_classes, num_classes), dtype=torch.float32)
        for i in range(num_classes):
            for j in range(num_classes):
                w[i, j] = ((i - j) ** 2) / ((num_classes - 1) ** 2)
        self.register_buffer("W", w)

    def forward(self, pred_probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        device = pred_probs.device
        N = pred_probs.size(0)
        if N == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)

        y_one_hot = F.one_hot(targets, num_classes=self.num_classes).float() if targets.ndim == 1 else targets.float()
        pred_probs = pred_probs / (pred_probs.sum(dim=1, keepdim=True) + self.eps)

        O = torch.matmul(pred_probs.t(), y_one_hot) / (N + self.eps)
        hist_pred = pred_probs.sum(dim=0, keepdim=True) / (N + self.eps)
        hist_true = y_one_hot.sum(dim=0, keepdim=True) / (N + self.eps)
        E = torch.matmul(hist_pred.t(), hist_true)

        W = self.W.to(device)
        return torch.sum(W * O) / (torch.sum(W * E) + self.eps)


# ------------------------------------------------------------------------------
# HEAD 1: Softmax + Categorical Cross-Entropy (Baseline)
# ------------------------------------------------------------------------------
class SoftmaxCEHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5):
        super(SoftmaxCEHead, self).__init__()
        self.fc = nn.Linear(in_features, num_classes)
        self.num_classes = num_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        rank_probs = torch.zeros(x.size(0), self.num_classes - 1, device=x.device)
        for k in range(self.num_classes - 1):
            rank_probs[:, k] = probs[:, k+1:].sum(dim=1)
        return {"logits": logits, "probs": probs, "preds": preds, "rank_probs": rank_probs}

    def loss_fn(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(outputs["logits"], targets)


# ------------------------------------------------------------------------------
# HEAD 2: Softmax + Continuous QWK Loss
# ------------------------------------------------------------------------------
class SoftmaxQWKHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5, qwk_weight: float = 1.0):
        super(SoftmaxQWKHead, self).__init__()
        self.fc = nn.Linear(in_features, num_classes)
        self.num_classes = num_classes
        self.qwk_weight = qwk_weight
        self.qwk_loss = ContinuousQWKLoss(num_classes=num_classes)

    def set_qwk_weight(self, weight: float):
        self.qwk_weight = float(weight)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        continuous_pred = torch.sum(probs * torch.arange(self.num_classes, device=x.device), dim=1)
        preds = torch.round(continuous_pred).clamp(0, self.num_classes - 1).long()
        rank_probs = torch.zeros(x.size(0), self.num_classes - 1, device=x.device)
        for k in range(self.num_classes - 1):
            rank_probs[:, k] = probs[:, k+1:].sum(dim=1)
        return {"logits": logits, "probs": probs, "preds": preds, "rank_probs": rank_probs}

    def loss_fn(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(outputs["logits"], targets)
        qwk = self.qwk_loss(outputs["probs"], targets)
        return ce_loss + self.qwk_weight * qwk


# ------------------------------------------------------------------------------
# HEAD 3: Consistent Rank Logits (CORAL)
# ------------------------------------------------------------------------------
class CORALHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5):
        super(CORALHead, self).__init__()
        self.num_classes = num_classes
        self.num_thresholds = num_classes - 1
        self.linear = nn.Linear(in_features, 1, bias=False)
        self.biases = nn.Parameter(torch.linspace(1.5, -1.5, self.num_thresholds))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        base_logit = self.linear(x)
        rank_logits = base_logit + self.biases
        raw_rank_probs = torch.sigmoid(rank_logits)
        preds = torch.sum(raw_rank_probs > 0.5, dim=1).long()

        B = x.size(0)
        probs = torch.zeros(B, self.num_classes, device=x.device)
        probs[:, 0] = 1.0 - raw_rank_probs[:, 0]
        for k in range(1, self.num_thresholds):
            probs[:, k] = F.relu(raw_rank_probs[:, k - 1] - raw_rank_probs[:, k])
        probs[:, -1] = raw_rank_probs[:, -1]
        probs = probs / (probs.sum(dim=1, keepdim=True) + 1e-7)

        return {"rank_logits": rank_logits, "rank_probs": raw_rank_probs, "probs": probs, "preds": preds}

    def loss_fn(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        rank_logits = outputs["rank_logits"]
        B = targets.size(0)
        binary_targets = torch.zeros(B, self.num_thresholds, device=targets.device)
        for k in range(self.num_thresholds):
            binary_targets[:, k] = (targets > k).float()
        return F.binary_cross_entropy_with_logits(rank_logits, binary_targets)


# ------------------------------------------------------------------------------
# HEAD 4: Conditional Ordinal Regression Neural Network (CORN)
# ------------------------------------------------------------------------------
class CORNHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5):
        super(CORNHead, self).__init__()
        self.num_classes = num_classes
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_features, self.num_thresholds)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        cond_logits = self.fc(x)
        cond_probs = torch.sigmoid(cond_logits)
        cum_probs = torch.cumprod(cond_probs, dim=1)
        preds = torch.sum(cum_probs > 0.5, dim=1).long()

        B = x.size(0)
        probs = torch.zeros(B, self.num_classes, device=x.device)
        probs[:, 0] = 1.0 - cum_probs[:, 0]
        for k in range(1, self.num_thresholds):
            probs[:, k] = F.relu(cum_probs[:, k - 1] - cum_probs[:, k])
        probs[:, -1] = cum_probs[:, -1]
        probs = probs / (probs.sum(dim=1, keepdim=True) + 1e-7)

        return {"cond_logits": cond_logits, "rank_probs": cum_probs, "probs": probs, "preds": preds}

    def loss_fn(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        cond_logits = outputs["cond_logits"]
        loss = 0.0
        valid_tasks = 0
        for k in range(self.num_thresholds):
            mask = (targets >= k)
            if mask.sum() > 0:
                task_logits = cond_logits[mask, k]
                task_targets = (targets[mask] > k).float()
                loss += F.binary_cross_entropy_with_logits(task_logits, task_targets)
                valid_tasks += 1
        return loss / max(1, valid_tasks)


# ------------------------------------------------------------------------------
# HEAD 5: Cumulative Link Model + QWK (Champion CLM_QWK)
# ------------------------------------------------------------------------------
class CumulativeLinkModelQWKHead(nn.Module):
    def __init__(self, in_features: int = 512, num_classes: int = 5, qwk_weight: float = 1.2):
        super(CumulativeLinkModelQWKHead, self).__init__()
        self.num_classes = num_classes
        self.num_cutpoints = num_classes - 1
        self.qwk_weight = qwk_weight
        self.target_qwk_weight = qwk_weight
        self.score_fc = nn.Linear(in_features, 1, bias=False)
        self.first_cutpoint = nn.Parameter(torch.tensor(-1.5))
        self.cutpoint_increments = nn.Parameter(torch.ones(self.num_cutpoints - 1) * -0.5)
        self.qwk_loss = ContinuousQWKLoss(num_classes=num_classes)

    def set_qwk_weight(self, weight: float):
        self.qwk_weight = float(weight)

    def _get_monotonic_cutpoints(self) -> torch.Tensor:
        increments = F.softplus(self.cutpoint_increments) + 1e-4
        cutpoints = [self.first_cutpoint]
        for i in range(len(increments)):
            cutpoints.append(cutpoints[-1] + increments[i])
        return torch.stack(cutpoints)

    def _cloglog_cdf(self, z: torch.Tensor) -> torch.Tensor:
        z_clamped = torch.clamp(z, min=-10.0, max=4.0)
        return 1.0 - torch.exp(-torch.exp(z_clamped))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = x.size(0)
        severity_score = self.score_fc(x).squeeze(1)
        cutpoints = self._get_monotonic_cutpoints()
        z = cutpoints.unsqueeze(0) - severity_score.unsqueeze(1)
        cum_probs = self._cloglog_cdf(z)

        probs = torch.zeros(B, self.num_classes, device=x.device)
        probs[:, 0] = cum_probs[:, 0]
        for k in range(1, self.num_cutpoints):
            probs[:, k] = F.relu(cum_probs[:, k - 1] - cum_probs[:, k])
        probs[:, -1] = F.relu(1.0 - cum_probs[:, -1])
        probs = probs / (probs.sum(dim=1, keepdim=True) + 1e-7)

        class_indices = torch.arange(self.num_classes, device=x.device, dtype=torch.float32)
        expected_grade = torch.sum(probs * class_indices, dim=1)
        preds = torch.round(expected_grade).clamp(0, self.num_classes - 1).long()

        rank_probs = torch.zeros(B, self.num_cutpoints, device=x.device)
        for k in range(self.num_cutpoints):
            rank_probs[:, k] = probs[:, k+1:].sum(dim=1)

        return {
            "severity_score": severity_score,
            "cutpoints": cutpoints,
            "probs": probs,
            "expected_grade": expected_grade,
            "preds": preds,
            "rank_probs": rank_probs
        }

    def loss_fn(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        probs = outputs["probs"]
        B = targets.size(0)
        target_probs = probs[torch.arange(B, device=targets.device), targets]
        nll_loss = -torch.log(torch.clamp(target_probs, min=1e-7)).mean()
        qwk = self.qwk_loss(probs, targets)
        return nll_loss + self.qwk_weight * qwk


# ------------------------------------------------------------------------------
# Unified ResNet50AblationModel Wrapper
# ------------------------------------------------------------------------------
class ResNet50AblationModel(nn.Module):
    """Unified End-to-End Model wrapping fresh ResNet-50 backbone with any of the 5 Heads."""
    def __init__(self, head_type: str = "CLM_QWK", pretrained: bool = True, num_classes: int = 5):
        super(ResNet50AblationModel, self).__init__()
        self.head_type = head_type
        self.backbone = ResNet50Backbone(pretrained=pretrained, latent_dim=512)

        if head_type == "Softmax_CE":
            self.head = SoftmaxCEHead(in_features=512, num_classes=num_classes)
        elif head_type == "Softmax_QWK":
            self.head = SoftmaxQWKHead(in_features=512, num_classes=num_classes, qwk_weight=1.0)
        elif head_type == "CORAL":
            self.head = CORALHead(in_features=512, num_classes=num_classes)
        elif head_type == "CORN":
            self.head = CORNHead(in_features=512, num_classes=num_classes)
        elif head_type == "CLM_QWK":
            self.head = CumulativeLinkModelQWKHead(in_features=512, num_classes=num_classes, qwk_weight=1.2)
        else:
            raise ValueError(f"Unsupported head_type: {head_type}")

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        latent = self.backbone(x)
        outputs = self.head(latent)
        outputs["latent"] = latent
        return outputs

    def compute_loss(self, outputs: Dict[str, torch.Tensor], targets: torch.Tensor) -> torch.Tensor:
        return self.head.loss_fn(outputs, targets)

## 4. Clinical Metrics & PyTorch Training Engine

Defines clinical statistical metrics (QWK, Within-1-Grade accuracy, Referable DR sensitivity/specificity at 80/90/95% fixed operating points, monotonicity violation percentage) and the high-performance training engine.

In [ ]:
# ==============================================================================
# 4.1 CLINICAL METRICS CALCULATORS & TRAINER ENGINE
# ==============================================================================
def rank_monotonicity_violations(rank_probs: np.ndarray) -> float:
    """Calculates proportion of samples violating non-increasing rank probability constraints."""
    if rank_probs is None or len(rank_probs) == 0:
        return 0.0
    rank_probs = np.asarray(rank_probs, dtype=np.float64)
    if rank_probs.ndim == 1:
        rank_probs = rank_probs.reshape(1, -1)
    diffs = np.diff(rank_probs, axis=1)
    violations = np.any(diffs > 1e-4, axis=1)
    return float(np.mean(violations))


def calculate_referable_sensitivity_at_specificity(
    y_true: np.ndarray,
    referable_probs: np.ndarray,
    target_specificities: Tuple[float, ...] = (0.80, 0.90, 0.95),
) -> Dict[str, float]:
    y_true_ref = (np.asarray(y_true, dtype=int) >= 2).astype(int)
    referable_probs = np.asarray(referable_probs, dtype=float)
    results: Dict[str, float] = {}

    n_pos = int(y_true_ref.sum())
    n_neg = int((1 - y_true_ref).sum())
    if n_pos == 0 or n_neg == 0:
        results["Referable_AUC"] = float("nan")
        for spec in target_specificities:
            results[f"Ref_Sensitivity_at_Spec{int(round(spec * 100))}"] = float("nan")
        return results

    fpr, tpr, _ = roc_curve(y_true_ref, referable_probs)
    results["Referable_AUC"] = float(auc(fpr, tpr))
    specificity = 1.0 - fpr

    for spec in target_specificities:
        valid_idx = np.where(specificity >= spec)[0]
        sens_at_spec = float(tpr[valid_idx].max()) if len(valid_idx) > 0 else 0.0
        results[f"Ref_Sensitivity_at_Spec{int(round(spec * 100))}"] = sens_at_spec

    return results


def calculate_dr_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    num_classes: int = 5,
    y_probs: Optional[np.ndarray] = None,
    rank_probs: Optional[np.ndarray] = None,
    target_specificities: Tuple[float, ...] = (0.80, 0.90, 0.95)
) -> Dict[str, Any]:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    qwk = float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))
    if np.isnan(qwk):
        qwk = 0.0

    acc = float(accuracy_score(y_true, y_pred))
    within_1 = float(np.mean(np.abs(y_true - y_pred) <= 1))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    per_grade_recall = {}
    for c in range(num_classes):
        denom = cm[c, :].sum()
        recall = float(cm[c, c] / denom) if denom > 0 else 0.0
        per_grade_recall[f"Recall_Grade_{c}"] = recall

    y_true_ref = (y_true >= 2).astype(int)
    y_pred_ref = (y_pred >= 2).astype(int)
    cm_ref = confusion_matrix(y_true_ref, y_pred_ref, labels=[0, 1])
    tn, fp, fn, tp = cm_ref.ravel() if cm_ref.size == 4 else (0, 0, 0, 0)

    ref_sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    ref_specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

    result: Dict[str, Any] = {
        "QWK": qwk,
        "Accuracy": acc,
        "Within_1_Grade_Acc": within_1,
        "Referable_Sensitivity": ref_sensitivity,
        "Referable_Specificity": ref_specificity,
        "Confusion_Matrix": cm.tolist(),
        **per_grade_recall
    }

    if y_probs is not None:
        y_probs = np.asarray(y_probs, dtype=float)
        referable_probs = y_probs[:, 2:].sum(axis=1) if y_probs.shape[1] >= 3 else y_probs[:, 1]
        spec_metrics = calculate_referable_sensitivity_at_specificity(
            y_true, referable_probs, target_specificities=target_specificities
        )
        result.update(spec_metrics)

    if rank_probs is not None:
        result["Monotonicity_Violations"] = rank_monotonicity_violations(rank_probs)
    elif y_probs is not None:
        result["Monotonicity_Violations"] = rank_monotonicity_violations(y_probs)
    else:
        result["Monotonicity_Violations"] = 0.0

    return result


class DRHeadTrainer:
    """End-to-End PyTorch Training Engine with Mixed Precision and ModelCheckpoint."""
    def __init__(
        self,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        lr_scheduler: Optional[Any] = None,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        use_amp: bool = True,
        grad_clip: float = 1.0,
        checkpoint_dir: str = CHECKPOINT_DIR_HEAD
    ):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.lr_scheduler = lr_scheduler
        self.device = device
        self.device_type = "cuda" if "cuda" in str(device) else "cpu"
        self.use_amp = use_amp and (self.device_type == "cuda")
        self.grad_clip = grad_clip
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.scaler = torch.amp.GradScaler(self.device_type, enabled=self.use_amp)

    def train_epoch(self, dataloader: DataLoader) -> Dict[str, float]:
        self.model.train()
        total_loss = 0.0
        all_preds, all_targets, all_probs, all_ranks = [], [], [], []

        for batch in dataloader:
            x = batch["image"].to(self.device, non_blocking=True)
            y = batch["label"].to(self.device, non_blocking=True)

            self.optimizer.zero_grad()
            with torch.amp.autocast(self.device_type, enabled=self.use_amp):
                outputs = self.model(x)
                loss = self.model.compute_loss(outputs, y)

            self.scaler.scale(loss).backward()
            if self.grad_clip > 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)

            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item() * len(y)
            all_preds.extend(outputs["preds"].detach().cpu().numpy())
            all_targets.extend(y.detach().cpu().numpy())
            if "probs" in outputs:
                all_probs.extend(outputs["probs"].detach().cpu().numpy())
            if "rank_probs" in outputs:
                all_ranks.extend(outputs["rank_probs"].detach().cpu().numpy())

        n_samples = len(all_targets)
        metrics = calculate_dr_metrics(
            all_targets, all_preds,
            y_probs=np.array(all_probs) if all_probs else None,
            rank_probs=np.array(all_ranks) if all_ranks else None
        )
        metrics["Loss"] = total_loss / max(1, n_samples)
        return metrics

    @torch.no_grad()
    def evaluate(self, dataloader: DataLoader) -> Tuple[Dict[str, Any], np.ndarray, np.ndarray]:
        self.model.eval()
        total_loss = 0.0
        all_preds, all_targets, all_probs, all_ranks = [], [], [], []

        for batch in dataloader:
            x = batch["image"].to(self.device, non_blocking=True)
            y = batch["label"].to(self.device, non_blocking=True)

            with torch.amp.autocast(self.device_type, enabled=self.use_amp):
                outputs = self.model(x)
                loss = self.model.compute_loss(outputs, y)

            total_loss += loss.item() * len(y)
            all_preds.extend(outputs["preds"].cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            if "probs" in outputs:
                all_probs.extend(outputs["probs"].cpu().numpy())
            if "rank_probs" in outputs:
                all_ranks.extend(outputs["rank_probs"].cpu().numpy())

        n_samples = len(all_targets)
        probs_np = np.array(all_probs) if all_probs else None
        ranks_np = np.array(all_ranks) if all_ranks else None
        metrics = calculate_dr_metrics(all_targets, all_preds, y_probs=probs_np, rank_probs=ranks_np)
        metrics["Loss"] = total_loss / max(1, n_samples)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return metrics, np.array(all_targets), np.array(all_preds)

    def fit(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        test_loader: Optional[DataLoader] = None,
        epochs: int = 12,
        head_name: str = "CLM_QWK",
        patience_early_stopping: int = 5
    ) -> Dict[str, Any]:
        best_val_qwk = -1.0
        best_val_loss = float("inf")
        epochs_no_improve = 0
        best_checkpoint_path = os.path.join(self.checkpoint_dir, f"{head_name.lower()}_best.pth")

        head = getattr(self.model, "head", None)
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            if head is not None and hasattr(head, "set_qwk_weight") and hasattr(head, "target_qwk_weight"):
                progress = min(1.0, epoch / max(1, QWK_WARMUP_EPOCHS))
                head.set_qwk_weight(progress * head.target_qwk_weight)

            train_metrics = self.train_epoch(train_loader)
            val_metrics, _, _ = self.evaluate(val_loader)
            elapsed = time.time() - t0

            val_loss = val_metrics["Loss"]
            val_qwk = val_metrics["QWK"]

            if self.lr_scheduler is not None:
                self.lr_scheduler.step()
            current_lr = self.optimizer.param_groups[0]['lr']

            logger.info(
                f"[{head_name} | Epoch {epoch:02d}/{epochs:02d} - {elapsed:.1f}s] "
                f"Train Loss: {train_metrics['Loss']:.4f} | Train QWK: {train_metrics['QWK']:.4f} || "
                f"Val Loss: {val_loss:.4f} | Val QWK: {val_qwk:.4f} | LR: {current_lr:.2e}"
            )

            if val_qwk > best_val_qwk:
                best_val_qwk = val_qwk
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": self.model.state_dict(),
                    "val_qwk": best_val_qwk,
                    "val_metrics": val_metrics
                }, best_checkpoint_path)
                logger.info(f"  -> Saved best checkpoint for {head_name} (Val QWK: {best_val_qwk:.4f})")

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience_early_stopping:
                    logger.info(f"  -> [EarlyStopping] Validation loss stagnant for {head_name}. Stopping at epoch {epoch}!")
                    break

        test_metrics = {}
        if test_loader is not None and os.path.exists(best_checkpoint_path):
            ckpt = torch.load(best_checkpoint_path, map_location=self.device, weights_only=False)
            self.model.load_state_dict(ckpt["model_state_dict"])
            test_metrics, _, _ = self.evaluate(test_loader)

        return {
            "head_name": head_name,
            "best_val_qwk": best_val_qwk,
            "best_checkpoint": best_checkpoint_path,
            "test_metrics": test_metrics
        }

## 5. Systematic 5-Round End-to-End Ablation Benchmark

This section executes the 5 independent end-to-end training loops comparing `Softmax_CE`, `Softmax_QWK`, `CORAL`, `CORN`, and `CLM_QWK` with fresh ResNet-50 backbones, evaluating on the 15% held-out test set and exporting structured summaries to CSV and JSON.

In [ ]:
# ==============================================================================
# 5. EXECUTE 5-ROUND END-TO-END HEAD ABLATION BENCHMARK
# ==============================================================================
head_ablation_results = {}
head_models_dict = {}
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n{'='*75}")
print("STARTING 5-ROUND END-TO-END HEAD ABLATION BENCHMARK (RESNET-50 BACKBONE)")
print(f"{'='*75}\n")

for round_idx, head_name in enumerate(EVALUATED_HEADS, 1):
    print(f"\n{'='*75}")
    print(f"ROUND {round_idx}/5: Training Fresh ResNet-50 Backbone + [{head_name}] End-to-End")
    print(f"{'='*75}")

    # 1. Instantiate fresh ResNet-50 + Head model (No shared weights, fair comparison)
    fresh_model = ResNet50AblationModel(head_type=head_name, pretrained=True, num_classes=5)
    optimizer = torch.optim.AdamW(fresh_model.parameters(), lr=1e-4, weight_decay=1e-3)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ABLATION_EPOCHS, eta_min=1e-6)

    trainer = DRHeadTrainer(
        model=fresh_model,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        device=device,
        use_amp=True,
        checkpoint_dir=CHECKPOINT_DIR_HEAD
    )

    # 2. Execute end-to-end training loop
    history = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=ABLATION_EPOCHS,
        head_name=head_name,
        patience_early_stopping=4
    )

    t_metrics = history["test_metrics"]
    head_ablation_results[head_name] = {
        "Head": head_name,
        "Val_QWK": history["best_val_qwk"],
        "Test_QWK": t_metrics.get("QWK", 0.0),
        "Accuracy": t_metrics.get("Accuracy", 0.0) * 100,
        "Within_1_Grade_Acc": t_metrics.get("Within_1_Grade_Acc", 0.0) * 100,
        "Referable_Sensitivity": t_metrics.get("Referable_Sensitivity", 0.0) * 100,
        "Referable_Specificity": t_metrics.get("Referable_Specificity", 0.0) * 100,
        "Referable_AUC": t_metrics.get("Referable_AUC", float("nan")),
        "Ref_Sens_Spec95": t_metrics.get("Ref_Sensitivity_at_Spec95", float("nan")) * 100,
        "Monotonicity_Violations": t_metrics.get("Monotonicity_Violations", 0.0) * 100,
        "Recall_Grade_0": t_metrics.get("Recall_Grade_0", 0.0) * 100,
        "Recall_Grade_1": t_metrics.get("Recall_Grade_1", 0.0) * 100,
        "Recall_Grade_2": t_metrics.get("Recall_Grade_2", 0.0) * 100,
        "Recall_Grade_3": t_metrics.get("Recall_Grade_3", 0.0) * 100,
        "Recall_Grade_4": t_metrics.get("Recall_Grade_4", 0.0) * 100,
        "Checkpoint_Path": history["best_checkpoint"]
    }
    head_models_dict[head_name] = fresh_model

    print(f"\n [RESULTS FOR {head_name}]:")
    print(f"  * Validation QWK               : {history['best_val_qwk']:.4f}")
    print(f"  * Held-Out Test QWK            : {t_metrics.get('QWK', 0.0):.4f}")
    print(f"  * Within-1-Grade Accuracy      : {head_ablation_results[head_name]['Within_1_Grade_Acc']:.1f}%")
    print(f"  * Rank Monotonicity Violations : {head_ablation_results[head_name]['Monotonicity_Violations']:.2f}%")
    print(f"  * Referable DR Sensitivity     : {head_ablation_results[head_name]['Referable_Sensitivity']:.1f}%")

# Save structured summary to CSV and JSON
df_heads = pd.DataFrame.from_dict(head_ablation_results, orient="index")
df_heads.to_csv(os.path.join(RESULTS_DIR_HEAD, "head_ablation_summary.csv"), index=False)
with open(os.path.join(RESULTS_DIR_HEAD, "head_ablation_summary.json"), "w", encoding="utf-8") as f:
    json.dump(head_ablation_results, f, indent=2)

print(f"\n{'='*75}")
print(f"5-Round Head Ablation Benchmark Completed & Exported to '{RESULTS_DIR_HEAD}'!")
print(f"{'='*75}\n")
print(df_heads[["Head", "Test_QWK", "Accuracy", "Within_1_Grade_Acc", "Monotonicity_Violations"]].to_string())

## 6. Publication-Quality Head Ablation Visualization Suite

Generates the complete 5-figure publication suite ($300\text{ DPI}$) comparing the 5 output head architectures on clinical and statistical criteria (QWK, within-1 accuracy, rank monotonicity violation rates, per-grade recall breakdown, continuous 1D latent score spectrum, and referable ROC curves).

In [ ]:
# ==============================================================================
# 6.0 PUBLICATION-READY FIGURE GENERATION FUNCTIONS (FIG 1 - 5)
# ==============================================================================
def plot_fig1_head_metric_comparison(results_dict: Dict[str, Any], save_path: Optional[str] = None) -> plt.Figure:
    """Renders Fig. 1: Multi-Metric Comparison (QWK, Within-1-Acc, Accuracy) across 5 Heads."""
    configure_publication_style()
    heads = list(results_dict.keys())
    qwk_vals = [results_dict[h]["Test_QWK"] for h in heads]
    w1_vals = [results_dict[h]["Within_1_Grade_Acc"] for h in heads]
    acc_vals = [results_dict[h]["Accuracy"] for h in heads]

    x = np.arange(len(heads))
    width = 0.26
    fig, ax = plt.subplots(figsize=(11.0, 5.5))

    r1 = ax.bar(x - width, qwk_vals, width, label="Test QWK", color=C_NAVY, edgecolor="#1e293b", lw=0.9, alpha=0.9)
    r2 = ax.bar(x, [w/100.0 for w in w1_vals], width, label="Within-1-Grade Acc (x100)", color=C_EMERALD, edgecolor="#1e293b", lw=0.9, alpha=0.9)
    r3 = ax.bar(x + width, [a/100.0 for a in acc_vals], width, label="Standard Accuracy (x100)", color=C_AMBER, edgecolor="#1e293b", lw=0.9, alpha=0.9)

    ax.set_ylabel("Performance Score", fontweight='bold')
    ax.set_title("Figure 1: Performance Comparison Across 5 Output Head Formulations (ResNet-50 Backbone)", fontweight='bold', pad=14)
    ax.set_xticks(x)
    ax.set_xticklabels(heads, fontweight='bold')
    ax.set_ylim(0.0, 1.05)
    ax.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#cbd5e1")
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    for i in range(len(heads)):
        ax.text(x[i] - width, qwk_vals[i] + 0.015, f"{qwk_vals[i]:.3f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold', color=C_NAVY)
        ax.text(x[i], w1_vals[i]/100.0 + 0.015, f"{w1_vals[i]:.1f}%", ha='center', va='bottom', fontsize=8.5, fontweight='bold', color=C_EMERALD)
        ax.text(x[i] + width, acc_vals[i]/100.0 + 0.015, f"{acc_vals[i]:.1f}%", ha='center', va='bottom', fontsize=8.5, fontweight='bold', color=C_AMBER)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


def plot_fig2_monotonicity_tradeoff(results_dict: Dict[str, Any], save_path: Optional[str] = None) -> plt.Figure:
    """Renders Fig. 2: Rank Monotonicity Violations vs Test QWK Tradeoff (Hypothesis H3)."""
    configure_publication_style()
    heads = list(results_dict.keys())
    qwk_vals = [results_dict[h]["Test_QWK"] for h in heads]
    viol_vals = [results_dict[h]["Monotonicity_Violations"] for h in heads]

    fig, ax1 = plt.subplots(figsize=(10.5, 5.5))
    ax2 = ax1.twinx()
    x = np.arange(len(heads))
    width = 0.35

    ax1.bar(x - width/2, qwk_vals, width, label="Quadratic Weighted Kappa (QWK)", color=C_TEAL, edgecolor="#1e293b", lw=0.9, alpha=0.9)
    ax2.bar(x + width/2, viol_vals, width, label="Monotonicity Violations (%)", color=C_CRIMSON, edgecolor="#1e293b", lw=0.9, alpha=0.9)

    ax1.set_ylabel("Test QWK Score (Higher is Better)", fontweight='bold', color=C_TEAL)
    ax2.set_ylabel("Monotonicity Violation Rate (%) (Lower is Better)", fontweight='bold', color=C_CRIMSON)
    ax1.set_title("Figure 2: Rank Monotonicity Violation Rate vs. QWK Generalization (Hypothesis $H_3$)", fontweight='bold', pad=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(heads, fontweight='bold')
    ax1.set_ylim(0.40, 0.95)
    ax2.set_ylim(-0.5, max(max(viol_vals) * 1.3, 15.0))

    for i in range(len(heads)):
        ax1.text(x[i] - width/2, qwk_vals[i] + 0.01, f"{qwk_vals[i]:.3f}", ha='center', va='bottom', fontsize=9.0, fontweight='bold', color=C_TEAL)
        ax2.text(x[i] + width/2, viol_vals[i] + 0.3, f"{viol_vals[i]:.1f}%", ha='center', va='bottom', fontsize=9.0, fontweight='bold', color=C_CRIMSON)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", frameon=True, facecolor="white", edgecolor="#cbd5e1")
    ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


def plot_fig3_per_grade_recall_breakdown(results_dict: Dict[str, Any], save_path: Optional[str] = None) -> plt.Figure:
    """Renders Fig. 3: Per-Grade Recall (Grade 0 to Grade 4) Breakdown across 5 Heads."""
    configure_publication_style()
    heads = list(results_dict.keys())
    grades = [f"Grade {g}" for g in range(5)]
    grade_colors = ["#3b82f6", C_ORANGE, "#eab308", C_CRIMSON, "#7e22ce"]

    fig, ax = plt.subplots(figsize=(12.0, 5.8))
    x = np.arange(len(heads))
    width = 0.15

    for g_idx in range(5):
        vals = [results_dict[h][f"Recall_Grade_{g_idx}"] for h in heads]
        ax.bar(x + (g_idx - 2) * width, vals, width, label=grades[g_idx], color=grade_colors[g_idx],
               edgecolor="#1e293b", lw=0.8, alpha=0.9)

    ax.set_ylabel("Sensitivity / Recall (%)", fontweight='bold')
    ax.set_title("Figure 3: Per-Grade Clinical Sensitivity Across Output Head Formulations", fontweight='bold', pad=14)
    ax.set_xticks(x)
    ax.set_xticklabels(heads, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#cbd5e1", title="Clinical Severity")
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


def plot_fig4_clm_latent_spectrum(clm_model: nn.Module, test_loader: DataLoader, device: str = "cuda", save_path: Optional[str] = None) -> plt.Figure:
    """Renders Fig. 4: 1D Latent Severity Score Spectrum s(x) and Monotonic Cutpoints of CLM_QWK."""
    configure_publication_style()
    clm_model.eval()
    all_scores, all_targets = [], []

    with torch.no_grad():
        for batch in test_loader:
            x = batch["image"].to(device)
            y = batch["label"].numpy()
            outputs = clm_model(x)
            scores = outputs["severity_score"].cpu().numpy()
            all_scores.extend(scores)
            all_targets.extend(y)

    scores_np = np.array(all_scores)
    targets_np = np.array(all_targets)
    thetas = clm_model.head._get_monotonic_cutpoints().detach().cpu().numpy().tolist()

    fig, ax = plt.subplots(figsize=(11, 5.5))
    grade_colors = ["#3b82f6", C_ORANGE, "#eab308", C_CRIMSON, "#7e22ce"]

    for g in range(5):
        g_scores = scores_np[targets_np == g]
        if len(g_scores) > 1:
            sns.kdeplot(g_scores, ax=ax, color=grade_colors[g], fill=True, alpha=0.35, lw=2.0, label=f"Grade {g}")

    for i, theta in enumerate(thetas):
        ax.axvline(theta, color=C_PURPLE, linestyle=":", lw=2.2, zorder=5)
        ax.text(theta, ax.get_ylim()[1] * 0.90, f"$\theta_{i+1} = {theta:.2f}$",
                ha="center", va="bottom", fontsize=10.0, fontweight="bold", color=C_PURPLE,
                bbox=dict(boxstyle="round,pad=0.2", fc="#faf5ff", ec=C_PURPLE, lw=1.0))

    ax.set_xlabel("1D Latent Severity Score: $s(\mathbf{x}) = \mathbf{w}^T \mathbf{z}$", fontsize=11.5, fontweight="bold")
    ax.set_ylabel("Sample Density", fontsize=11.5, fontweight="bold")
    ax.set_title("Figure 4: 1D Latent Severity Score Spectrum & Monotonic Cutpoints of Proposed CLM_QWK Head\n"
                 "Learned Continuum Proves Non-Arbitrary Distance Preservation Between Severity Levels", 
                 fontsize=12, fontweight="bold", pad=15)
    ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#cbd5e1", title="Clinical Severity")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


def plot_fig5_referable_roc_curves(head_models: Dict[str, nn.Module], test_loader: DataLoader, device: str = "cuda", save_path: Optional[str] = None) -> plt.Figure:
    """Renders Fig. 5: Referable DR ROC Curves across all 5 Output Heads on the Held-Out Test Set."""
    configure_publication_style()
    fig, ax = plt.subplots(figsize=(8.0, 6.2))
    head_colors = {"Softmax_CE": C_NAVY, "Softmax_QWK": C_AMBER, "CORAL": C_ORANGE, "CORN": C_GRAY, "CLM_QWK": C_TEAL}

    for head_name, model in head_models.items():
        model.eval()
        all_targets, all_probs = [], []
        with torch.no_grad():
            for batch in test_loader:
                x = batch["image"].to(device)
                y = batch["label"].numpy()
                outputs = model(x)
                probs = outputs["probs"].cpu().numpy()
                all_targets.extend(y)
                all_probs.extend(probs)

        y_true = np.array(all_targets)
        probs_np = np.array(all_probs)
        y_true_ref = (y_true >= 2).astype(int)
        ref_probs = probs_np[:, 2:].sum(axis=1) if probs_np.shape[1] >= 3 else probs_np[:, 1]

        if len(np.unique(y_true_ref)) > 1:
            fpr, tpr, _ = roc_curve(y_true_ref, ref_probs)
            roc_auc = float(auc(fpr, tpr))
            lw = 3.0 if head_name == "CLM_QWK" else 1.8
            ls = "-" if head_name == "CLM_QWK" else "--"
            ax.plot(fpr, tpr, color=head_colors.get(head_name, C_NAVY), lw=lw, linestyle=ls,
                    label=f"{head_name} (AUC = {roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], color="#94a3b8", linestyle=":", lw=1.2)
    ax.axvline(0.05, color="#cbd5e1", linestyle="--", lw=1.2)
    ax.text(0.06, 0.40, "95% Screening Specificity Gate (FPR = 5%)", rotation=90, va="center", color="#475569", fontsize=9.5)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11, fontweight="bold")
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11, fontweight="bold")
    ax.set_title("Figure 5: Referable DR ROC Curves Across 5 Output Heads (Held-Out Test Set)", fontsize=12, fontweight="bold", pad=15)
    ax.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#cbd5e1")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig

In [ ]:
# ==============================================================================
# 6.1 EXECUTE FULL FIGURE GENERATION SUITE & EXPORT TO RESULTS/FIGURES/
# ==============================================================================
print("\n" + "="*75)
print("GENERATING PUBLICATION-QUALITY HEAD ABLATION FIGURES (FIG 1 - 5)")
print("="*75 + "\n")

# 1. Figure 1: Multi-Metric Comparison
print("[Fig. 1] Generating Multi-Metric Performance Comparison Bar Chart...")
f1 = plot_fig1_head_metric_comparison(
    head_ablation_results,
    save_path=os.path.join(FIGURES_DIR, "figure_head_ablation_1_metrics.png")
)
plt.close(f1)

# 2. Figure 2: Monotonicity Violations vs QWK Tradeoff
print("[Fig. 2] Generating Monotonicity Violations vs QWK Tradeoff Chart...")
f2 = plot_fig2_monotonicity_tradeoff(
    head_ablation_results,
    save_path=os.path.join(FIGURES_DIR, "figure_head_ablation_2_monotonicity_tradeoff.png")
)
plt.close(f2)

# 3. Figure 3: Per-Grade Sensitivity Breakdown
print("[Fig. 3] Generating Per-Grade Sensitivity Breakdown Chart...")
f3 = plot_fig3_per_grade_recall_breakdown(
    head_ablation_results,
    save_path=os.path.join(FIGURES_DIR, "figure_head_ablation_3_per_grade_recall.png")
)
plt.close(f3)

# 4. Figure 4: 1D Latent Severity Score Spectrum & Monotonic Cutpoints (CLM_QWK)
if "CLM_QWK" in head_models_dict:
    print("[Fig. 4] Generating 1D Latent Severity Spectrum & Monotonic Cutpoints (CLM_QWK)...")
    f4 = plot_fig4_clm_latent_spectrum(
        head_models_dict["CLM_QWK"],
        test_loader,
        device=device,
        save_path=os.path.join(FIGURES_DIR, "figure_head_ablation_4_clm_latent_spectrum.png")
    )
    plt.close(f4)

# 5. Figure 5: Referable DR ROC Curves across 5 Heads
print("[Fig. 5] Generating Referable DR ROC Curves Comparison...")
f5 = plot_fig5_referable_roc_curves(
    head_models_dict,
    test_loader,
    device=device,
    save_path=os.path.join(FIGURES_DIR, "figure_head_ablation_5_roc_comparison.png")
)
plt.close(f5)

print(f"\n{'='*75}")
print(f"All 5 Head Ablation Publication Figures Successfully Generated and Saved in '{FIGURES_DIR}'!")
print(f"{'='*75}\n")